In [1]:

import sys
from pathlib import Path
BASE_PATH = Path.cwd().absolute()

sys.path.append(str(BASE_PATH))
sys.path.append(str(BASE_PATH / "Studies"))
sys.path.append(str(BASE_PATH / "Functions"))
sys.path.append(str(BASE_PATH / "Costum_Models"))

import Gandalf as Gan_model
import MLP as MLP_model
import Confinv
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import xgboost as xgb

import pytorch_tabnet
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import sqlalchemy as sa
import Results as res
import Get_Data as gd
import time
from sklearn.model_selection import train_test_split

import torch

from torch.utils.data import DataLoader
import torch.utils.data as data_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu=torch.device("cpu")
import pytorch_tabnet

print("Python:", sys.executable)
print("pytorch_tabnet path:", pytorch_tabnet.__file__)

device: cuda
gpu: NVIDIA GeForce RTX 4060 Laptop GPU
Python: /home/jrech/predictive-goods-receipt-scheduling/.venv/bin/python
pytorch_tabnet path: None


In [2]:
path="Database/DB_params.db"
engine = sa.create_engine("sqlite:///" + path)
engine.connect()


path_train="Training_Test_Datensatz/Training_Datensatz.xlsx"
path_test="Training_Test_Datensatz/Test_Datensatz.xlsx"


Train= pd.read_excel(path_train)
Test= pd.read_excel(path_test)

X_test_raw = Test.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_test_raw = Test["verzoegerung_bin_5"]
w_test_raw = Test["w_time"]

X_train_raw = Train.drop(columns=["verzoegerung_bin_5","w_time","Verzoegerungstage"])
y_train_raw = Train["verzoegerung_bin_5"]
w_train_raw = Train["w_time"]



In [3]:
conf=Confinv.ConfidenceIntervalCalculator()

In [4]:
Model="Logistic Regression"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()


if best_params["class_weight"]=="balanced":
    best_params["class_weight"] = "balanced"
else:
    best_params["class_weight"] = None

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
model = LogisticRegression(
    solver=best_params["solver"],
    C=best_params["C"],
    max_iter=best_params["max_iter"],
    class_weight=best_params["class_weight"],
    random_state=42
)



start=time.time()
model.fit(X_train, y_train, sample_weight=w_train)
end=time.time()


X_test = np.nan_to_num(X_test, nan=0)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Results for Logistic Regression saved successfully at 2026-06-03 13:36.


In [5]:

Model="CatBoost"
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

final_model = CatBoostClassifier(**best_params, allow_writing_files=False)

start=time.time()
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    early_stopping_rounds=50,
    verbose=False
)
end=time.time()
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)
traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred.flatten(), y_proba ,y_test, w_test, training_time=traing_time)



Results for CatBoost saved successfully at 2026-06-03 13:37.


In [6]:

Model="Gandalf"
epochs=100
loss_fn = nn.CrossEntropyLoss(reduction="none")
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)


best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

model = Gan_model.Model(
                    input_dim=X_train.shape[1],
                    gflu_stages=best_params["gflu_stages"],
                    gflu_dropout=best_params["gflu_dropout"],
                    gflu_feature_init_sparsity=best_params["gflu_feature_init_sparsity"],
                    learnable_sparsity=True,
                    )

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
sheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, sheduler, loss_fn)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)


conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)

traing_time=end-start
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Epochs 1, Loss: 1.4440, Val Loss: 1.3622, Val Acc: 0.4432
Epochs 2, Loss: 1.3034, Val Loss: 1.3100, Val Acc: 0.4653
Epochs 3, Loss: 1.2403, Val Loss: 1.2709, Val Acc: 0.4836
Epochs 4, Loss: 1.1936, Val Loss: 1.2318, Val Acc: 0.4955
Epochs 5, Loss: 1.1479, Val Loss: 1.1920, Val Acc: 0.5150
Epochs 6, Loss: 1.1112, Val Loss: 1.1711, Val Acc: 0.5362
Epochs 7, Loss: 1.0785, Val Loss: 1.1334, Val Acc: 0.5457
Epochs 8, Loss: 1.0421, Val Loss: 1.1154, Val Acc: 0.5459
Epochs 9, Loss: 1.0118, Val Loss: 1.0781, Val Acc: 0.5655
Epochs 10, Loss: 0.9804, Val Loss: 1.0542, Val Acc: 0.5776
Epochs 11, Loss: 0.9462, Val Loss: 1.0430, Val Acc: 0.5772
Epochs 12, Loss: 0.9217, Val Loss: 1.0202, Val Acc: 0.5974
Epochs 13, Loss: 0.9001, Val Loss: 1.0017, Val Acc: 0.6178
Epochs 14, Loss: 0.8806, Val Loss: 0.9791, Val Acc: 0.6089
Epochs 15, Loss: 0.8566, Val Loss: 0.9664, Val Acc: 0.6094
Epochs 16, Loss: 0.8373, Val Loss: 0.9745, Val Acc: 0.6151
Epochs 17, Loss: 0.8224, Val Loss: 0.9462, Val Acc: 0.6356
Epochs

In [7]:
Model="LightGBM"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer"))
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)
best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

model = LGBMClassifier(**best_params)

start=time.time()
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    sample_weight=w_tr,
    eval_sample_weight=[w_val]
)
end=time.time()
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
traing_time=end-start
conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Results for LightGBM saved successfully at 2026-06-03 13:42.


In [ ]:

Model="Multilayer Perceptron"
epochs=100
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()
X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw, scaler=best_params.pop("scaler"), encoder=best_params.pop("encoder") , periodic_transformer=best_params.pop("periodic_transformer") , feature_output="MLP")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")

batch_size=best_params.pop("batch_size")

train_dataset = data_utils.TensorDataset(X_tr, y_tr, w_tr)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = data_utils.TensorDataset(X_val, y_val, w_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


model=MLP_model.Model(input_dim=X_train.shape[1])

optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=best_params.pop("weight_decay"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, min_lr=1e-6)


start=time.time()
accuracay_plot=model.fit(train_dataloader, val_dataloader,epochs, optimizer, scheduler)
end=time.time()

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)

Fold 1, Loss: 1.5500, Val Loss: 1.2997, Val Acc: 0.4782
Fold 2, Loss: 1.3581, Val Loss: 1.2350, Val Acc: 0.5141
Fold 3, Loss: 1.3087, Val Loss: 1.2064, Val Acc: 0.5153
Fold 4, Loss: 1.2574, Val Loss: 1.1772, Val Acc: 0.5253
Fold 5, Loss: 1.2316, Val Loss: 1.1512, Val Acc: 0.5340
Fold 6, Loss: 1.1973, Val Loss: 1.1291, Val Acc: 0.5495
Fold 7, Loss: 1.1826, Val Loss: 1.1122, Val Acc: 0.5516
Fold 8, Loss: 1.1561, Val Loss: 1.0956, Val Acc: 0.5652
Fold 9, Loss: 1.1357, Val Loss: 1.0744, Val Acc: 0.5631
Fold 10, Loss: 1.1214, Val Loss: 1.0623, Val Acc: 0.5691
Fold 11, Loss: 1.0965, Val Loss: 1.0423, Val Acc: 0.5910
Fold 12, Loss: 1.0808, Val Loss: 1.0267, Val Acc: 0.5910
Fold 13, Loss: 1.0719, Val Loss: 1.0158, Val Acc: 0.5933
Fold 14, Loss: 1.0534, Val Loss: 1.0030, Val Acc: 0.6076
Fold 15, Loss: 1.0399, Val Loss: 0.9946, Val Acc: 0.6056
Fold 16, Loss: 1.0288, Val Loss: 0.9799, Val Acc: 0.6134
Fold 17, Loss: 1.0169, Val Loss: 0.9694, Val Acc: 0.6202
Fold 18, Loss: 1.0111, Val Loss: 0.9606,

In [ ]:
Model="TabNet"
best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

X_train, X_test, y_train, y_test, w_train, w_test, cat_idxs, cat_dims = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler="none", encoder="ordinal", periodic_transformer=False, feature_output="TabNet")
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)

scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
batch_size = best_params.pop("batch_size")
optimizer_params=best_params.pop("lr")
step_size=best_params.pop("step_size")
scheduler_gamma=best_params.pop("scheduler_gamma")

best_params.pop("model_name")
best_params.pop("best_value")
best_params.pop("saved_at")



model = TabNetClassifier(
    **best_params,
    optimizer_params=dict(lr=optimizer_params),
    scheduler_params=dict(step_size=step_size, gamma=scheduler_gamma),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    seed=42,
)

start=time.time()

model.fit(
    X_train=X_tr,
    y_train=y_tr,
    weights=w_tr,
    eval_set=[(X_val, y_val)],
    eval_metric=['accuracy'],
    max_epochs=300,
    patience=100,
    batch_size=batch_size,
)
end=time.time()


X_train = np.nan_to_num(X_train, nan=0)
X_test = np.nan_to_num(X_test, nan=0)

y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)

traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)
res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)


/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.18881 | val_0_accuracy: 0.21031 |  0:00:01s
epoch 1  | loss: 2.11058 | val_0_accuracy: 0.17518 |  0:00:02s
epoch 2  | loss: 1.74931 | val_0_accuracy: 0.23038 |  0:00:03s
epoch 3  | loss: 1.70723 | val_0_accuracy: 0.25319 |  0:00:04s
epoch 4  | loss: 1.60878 | val_0_accuracy: 0.23403 |  0:00:05s
epoch 5  | loss: 1.60799 | val_0_accuracy: 0.26095 |  0:00:09s
epoch 6  | loss: 1.52243 | val_0_accuracy: 0.21898 |  0:00:10s
epoch 7  | loss: 1.62623 | val_0_accuracy: 0.26323 |  0:00:12s
epoch 8  | loss: 1.50816 | val_0_accuracy: 0.23859 |  0:00:13s
epoch 9  | loss: 1.52163 | val_0_accuracy: 0.2167  |  0:00:15s
epoch 10 | loss: 1.42293 | val_0_accuracy: 0.23677 |  0:00:16s
epoch 11 | loss: 1.35785 | val_0_accuracy: 0.22673 |  0:00:18s
epoch 12 | loss: 1.35496 | val_0_accuracy: 0.26962 |  0:00:19s
epoch 13 | loss: 1.35694 | val_0_accuracy: 0.23586 |  0:00:20s
epoch 14 | loss: 1.32564 | val_0_accuracy: 0.24681 |  0:00:21s
epoch 15 | loss: 1.34858 | val_0_accuracy: 0.28193 |  0

/home/jrech/predictive-goods-receipt-scheduling/.venv/lib/python3.12/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Results for TabNet saved successfully at 2026-06-03 13:01.


In [ ]:

Model="XGBoost"

best_params=pd.read_sql(Model, engine).iloc[0].to_dict()

scaler = best_params.pop("scaler")
encoder = best_params.pop("encoder")
periodic_transformer = best_params.pop("periodic_transformer")
number_ouf_boost_rounds = best_params.pop("num_boost_round")

X_train, X_test, y_train, y_test, w_train, w_test = gd.preprocess_split(X_train_raw, X_test_raw, y_train_raw, y_test_raw, w_train_raw, w_test_raw,scaler=scaler,encoder=encoder,periodic_transformer=periodic_transformer)
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split( X_train, y_train, w_train,test_size=0.2,stratify=y_train,random_state=42)



dtrain = xgb.DMatrix(X_tr, label=y_tr, weight=w_tr)
dval   = xgb.DMatrix(X_val, label=y_val, weight=w_val)
dtest  = xgb.DMatrix(X_test, label=y_test, weight=w_test)

start=time.time()
bst = xgb.train(
    best_params,
    dtrain,
    num_boost_round=number_ouf_boost_rounds,
    evals=[(dval, "val")],
    early_stopping_rounds=30,
    verbose_eval=False
)
end=time.time()
traing_time=end-start
y_proba = bst.predict(dtest, output_margin=False)
y_pred = y_proba.argmax(axis=1)
end=time.time()


traing_time=end-start

conf.bootstrap_confidence_intervals(
    model_name=Model,
    y_true=y_test,
    y_pred=y_pred,
    sample_weight=w_test
)





res.save_frames(Model, y_pred, y_proba ,y_test, w_test, training_time=traing_time)





Results for XGBoost saved successfully at 2026-06-03 13:06.


In [ ]:
sqlite_path = "Database/DB_results.db"
engine = sa.create_engine("sqlite:///" + sqlite_path)
engine.connect()
df=conf.bootstrap_results.round(4)
df.to_sql("ConfidenceIntervals", con=engine, if_exists="replace", index=False)


50